# Lab 23: Neural Networks as Matrix Machines

This lab is a deeper computational companion to Chapter 23. You will build neural-network ideas from scratch using NumPy: neurons, layers, activations, forward passes, batch computation, decision boundaries, hidden representations, softmax, gradient descent, and a small nonlinear network.

The goal is not to use a deep-learning library. The goal is to see the linear algebra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1/(1+np.exp(-z))

def softmax(scores, axis=-1):
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    e = np.exp(shifted)
    return e/e.sum(axis=axis, keepdims=True)

## 1. One neuron: dot product plus bias

A neuron first computes $z=w\cdot x+b$, then applies an activation $h=\sigma(z)$.

In [ ]:
x = np.array([2.0, 3.0, -1.0])
w = np.array([0.5, -1.0, 2.0])
b = 1.0

z = w @ x + b
h = relu(z)
print('z =', z)
print('ReLU(z) =', h)

### Student task
Change the vector $w$ so that the same input activates the neuron strongly. What geometric relationship between $w$ and $x$ makes the dot product large?

## 2. Activation functions

Activation functions are nonlinear gates. Compare ReLU, sigmoid, and tanh.

In [ ]:
z_grid = np.linspace(-6, 6, 600)
plt.figure(figsize=(8,4.5))
plt.plot(z_grid, relu(z_grid), label='ReLU')
plt.plot(z_grid, sigmoid(z_grid), label='sigmoid')
plt.plot(z_grid, np.tanh(z_grid), label='tanh')
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel('z')
plt.ylabel('activation')
plt.title('Activation Functions')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. One layer: many neurons at once

A layer computes $h=\sigma(Wx+b)$. Each row of $W$ is a neuron.

In [ ]:
x = np.array([2.0, 3.0])
W = np.array([[1, 0], [0, 1], [1, -1], [-1, 1]], dtype=float)
b = np.array([0, 0, 1, 1], dtype=float)
z = W @ x + b
h = relu(z)
print('pre-activation z =', z)
print('hidden representation h =', h)

## 4. Batch computation

If the rows of $X$ are examples, then $Z=XW^T+\mathbf{1}b^T$.

In [ ]:
X = np.array([[2,3], [1,1], [4,0.5], [-1,2], [0,-2]], dtype=float)
Z = X @ W.T + b
H = relu(Z)
print('Z shape:', Z.shape)
print(H)

## 5. Visualizing a ReLU neuron in the plane

A neuron changes behavior across the line $w\cdot x+b=0$.

In [ ]:
x1 = np.linspace(-3, 3, 300)
x2 = np.linspace(-3, 3, 300)
X1, X2 = np.meshgrid(x1, x2)
w = np.array([1.0, -0.7])
b = -0.2
Z = w[0]*X1 + w[1]*X2 + b
A = relu(Z)
plt.figure(figsize=(6,5))
plt.contourf(X1, X2, A, levels=25)
plt.contour(X1, X2, Z, levels=[0], linewidths=2)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('One ReLU Neuron')
plt.colorbar(label='activation')
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

## 6. A hidden layer creates a new representation

Now map two-dimensional points into three hidden features.

In [ ]:
rng = np.random.default_rng(1)
points = rng.normal(size=(500,2))
W_hidden = np.array([[1, 1], [1, -1], [-1, 0.5]], dtype=float)
b_hidden = np.array([-0.3, 0.0, 0.2])
H = relu(points @ W_hidden.T + b_hidden)
plt.figure(figsize=(6,5))
plt.scatter(H[:,0], H[:,1], c=H[:,2], s=18, alpha=0.8)
plt.xlabel('hidden feature 1')
plt.ylabel('hidden feature 2')
plt.title('Hidden Representation of Random Points')
plt.colorbar(label='hidden feature 3')
plt.grid(True, alpha=0.3)
plt.show()

## 7. XOR: nonlinear representation solves a nonlinear pattern

The XOR labels are not separable by one line. A hidden layer can create useful intermediate features.

In [ ]:
X_xor = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y_xor = np.array([0,1,1,0])
W1 = np.array([[1,1], [1,1]], dtype=float)
b1 = np.array([0, -1], dtype=float)
W2 = np.array([[1, -2]], dtype=float)
b2 = np.array([0.0])
H = relu(X_xor @ W1.T + b1)
scores = H @ W2.T + b2
print(np.c_[X_xor, y_xor, H, scores])

In [ ]:
plt.figure(figsize=(5,5))
for label, marker in [(0, 'o'), (1, 's')]:
    mask = y_xor == label
    plt.scatter(X_xor[mask,0], X_xor[mask,1], s=130, marker=marker, label=f'class {label}')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('XOR in Input Space')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(5,5))
for label, marker in [(0, 'o'), (1, 's')]:
    mask = y_xor == label
    plt.scatter(H[mask,0], H[mask,1], s=130, marker=marker, label=f'class {label}')
plt.xlabel('$h_1$')
plt.ylabel('$h_2$')
plt.title('XOR After Hidden Features')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Softmax and cross-entropy

Softmax turns scores into probabilities. Cross-entropy punishes low probability on the true class.

In [ ]:
scores = np.array([[1.0, 2.0, 0.5], [3.0, 0.5, 1.0]])
probs = softmax(scores, axis=1)
true_classes = np.array([1, 0])
losses = -np.log(probs[np.arange(len(true_classes)), true_classes])
print('probabilities:
', probs)
print('losses:', losses)
print('mean loss:', losses.mean())

## 9. Training a linear model by gradient descent

This is the simplest case of learning parameters to reduce loss.

In [ ]:
np.random.seed(8)
X = np.linspace(-3, 3, 100)
y = 2.2*X - 0.7 + 0.5*np.random.randn(100)
w = 0.0
b = 0.0
eta = 0.04
losses = []
for step in range(300):
    yhat = w*X + b
    error = yhat-y
    losses.append(np.mean(error**2))
    w -= eta*2*np.mean(error*X)
    b -= eta*2*np.mean(error)
print(w, b)
plt.figure(figsize=(7,4))
plt.plot(losses)
plt.title('Loss Decreases')
plt.xlabel('step')
plt.ylabel('MSE')
plt.grid(True, alpha=0.3)
plt.show()
plt.figure(figsize=(7,4))
plt.scatter(X, y, s=20, alpha=0.6)
plt.plot(X, w*X+b, linewidth=2)
plt.title('Learned Linear Fit')
plt.grid(True, alpha=0.3)
plt.show()

## 10. Training a tiny ReLU network from scratch

Now fit a nonlinear curve with one hidden layer. This is backpropagation in NumPy, written explicitly.

In [ ]:
np.random.seed(4)
X = np.linspace(-2, 2, 160).reshape(-1,1)
y = (np.sin(3*X[:,0]) + 0.2*np.random.randn(160)).reshape(-1,1)

hidden = 16
W1 = 0.6*np.random.randn(hidden,1)
b1 = np.zeros(hidden)
W2 = 0.6*np.random.randn(1,hidden)
b2 = np.zeros(1)
eta = 0.02
losses = []

for step in range(2000):
    Z1 = X @ W1.T + b1
    H = relu(Z1)
    Yhat = H @ W2.T + b2
    E = Yhat - y
    loss = np.mean(E**2)
    losses.append(loss)
    dY = 2*E/len(X)
    dW2 = dY.T @ H
    db2 = dY.sum(axis=0)
    dH = dY @ W2
    dZ1 = dH * (Z1 > 0)
    dW1 = dZ1.T @ X
    db1 = dZ1.sum(axis=0)
    W2 -= eta*dW2
    b2 -= eta*db2
    W1 -= eta*dW1
    b1 -= eta*db1

plt.figure(figsize=(7,4))
plt.plot(losses)
plt.xlabel('step')
plt.ylabel('MSE')
plt.title('Training a Tiny ReLU Network')
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(7,4))
plt.scatter(X[:,0], y[:,0], s=18, alpha=0.55, label='data')
plt.plot(X[:,0], Yhat[:,0], linewidth=2, label='network')
plt.xlabel('x')
plt.ylabel('y')
plt.title('A One-Hidden-Layer Network Fits a Nonlinear Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 11. Parameter counts

For a fully connected layer from dimension $n$ to dimension $m$, the number of trainable parameters is $mn+m$.

In [ ]:
def dense_params(input_dim, output_dim):
    return input_dim*output_dim + output_dim

for input_dim, output_dim in [(784,128), (3072,200), (768,3072), (3072,768)]:
    print(f'{input_dim} -> {output_dim}: {dense_params(input_dim, output_dim):,} parameters')

## 12. High-dimensional hidden representations

Random projections followed by ReLU can change similarity and sparsity. This is a simple high-dimensional experiment.

In [ ]:
rng = np.random.default_rng(12)
N, d, m = 400, 100, 500
X = rng.normal(size=(N,d))
W = rng.normal(size=(m,d))/np.sqrt(d)
b = rng.normal(scale=0.1, size=m)
H = relu(X @ W.T + b)

input_norms = np.linalg.norm(X, axis=1)
hidden_norms = np.linalg.norm(H, axis=1)
sparsity = (H == 0).mean(axis=1)

plt.figure(figsize=(7,4))
plt.hist(hidden_norms, bins=30, alpha=0.8)
plt.xlabel('hidden vector norm')
plt.ylabel('count')
plt.title('Norms of High-Dimensional Hidden Representations')
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(7,4))
plt.hist(sparsity, bins=30, alpha=0.8)
plt.xlabel('fraction of zero coordinates')
plt.ylabel('count')
plt.title('ReLU Creates Sparse Hidden Representations')
plt.grid(True, alpha=0.3)
plt.show()

## Final reflection

Write a short paragraph answering: In what sense is a neural network a matrix machine, and in what sense is it more than a matrix machine?